# Asian CDM Landfill Gas & Engineering Dataset — quick start

This notebook loads the v1.0 release tables and shows the main coverage patterns.

> **Important:** this is a CDM-selected site sample, not a national inventory. It records recovered/collected landfill gas, not total methane generated or emitted. Do **not** use it for country-level methane totals.

In [ ]:
from pathlib import Path

import pandas as pd

root = Path.cwd()
if not (root / "release" / "v1.0" / "data").exists():
    root = root.parent
data = root / "release" / "v1.0" / "data"
print(data)

In [ ]:
sites = pd.read_csv(data / "sites_108.csv")
lfg = pd.read_csv(data / "lfg_measurements.csv")
waste = pd.read_csv(data / "waste_inflow_annual.csv")
composition = pd.read_csv(data / "waste_composition.csv")
climate = pd.read_csv(data / "climate_annual.csv")

for name, frame in {
    "sites": sites,
    "lfg": lfg,
    "waste": waste,
    "composition": composition,
    "climate": climate,
}.items():
    print(f"{name:12} {frame.shape[0]:5} rows x {frame.shape[1]:3} columns")

## Coverage by region and evidence tier

Tier A has at least one metered landfill-gas period. Tier B has design/operation documentation but no metered gas series.

In [ ]:
sites.groupby(["region", "evidence_tier"]).size().unstack(fill_value=0)

In [ ]:
sites.groupby("country").size().sort_values(ascending=False).to_frame("sites")

## Measured landfill gas

There are only 48 measured monitoring rows across 22 sites. Most sites have 1–3 rows; one site has 10.

In [ ]:
lfg.groupby("landfill_id").size().sort_values(ascending=False).to_frame("rows")

In [ ]:
lfg[[
    "observation_id", "landfill_id", "country", "year", "period_start", "period_end",
    "lfg_total_volume", "lfg_volume_unit", "methane_collected_tonnes",
    "qa_outlier", "validation_status",
]].head(20)

### Units are mixed on purpose

The release keeps `m3` and `Nm3` as reported. It does not convert between them because the source tables do not consistently report the temperature and pressure needed for a defensible conversion. Always group or filter by `lfg_volume_unit` before summing.

In [ ]:
lfg.groupby("lfg_volume_unit")["lfg_total_volume"].agg(["count", "sum"])

## Annual waste inflow and composition

The annual inflow table is a site-year series. The site table also carries one site-level reference value for convenience; do not confuse the two.

In [ ]:
waste.groupby("landfill_id")["year"].agg(["min", "max", "count"]).sort_values("count", ascending=False).head(15)

In [ ]:
# 16 of 42 site composition records do not sum to 1.0 and are flagged, not renormalised.
composition.drop_duplicates("landfill_id")[[
    "landfill_id", "composition_sum", "composition_is_complete"
]].sort_values("composition_sum").head(20)

## Climate covariates

NASA POWER provides annual temperature, relative humidity and precipitation for 2010–2025.

In [ ]:
climate.groupby("landfill_id").size().describe()

## Why this cannot produce country totals

A country estimate needs a complete national site universe, representative sampling, annual activity data, IPCC FOD parameters, fugitive-emission measurements and an uncertainty framework. This release has a CDM-selected subset, sparse measured years and no complete mass balance.

The code below shows the structural limitation: 86 of 108 sites have no metered LFG at all.

In [ ]:
sites.groupby(["country", "has_metered_lfg"]).size().unstack(fill_value=0)

## Next steps

- Read `release/v1.0/QA_REPORT.md` before analysis.
- Exclude the unverified `cdm_839` 910.8 million m³ row unless you verify it against the source table.
- Filter by `lfg_volume_unit` before summing volumes.
- Use `waste_inflow_annual.csv` for time series and `sites_108.csv` for site attributes.
- Cite the dataset and the underlying sources (see `CITATION.cff` and `SOURCES.md`).